# 03 — Data Cleaning

Notebook này thực hiện Giai đoạn 3 cho dự án:

**Phân cụm tin tuyển dụng tại Việt Nam dựa trên mô tả công việc, yêu cầu ứng viên và đặc điểm tuyển dụng**

Mục tiêu:

1. Load `raw_data_train.csv` và `raw_data_test.csv`.
2. Kiểm tra tính độc lập train/test.
3. Làm sạch text và categorical columns.
4. Chuẩn hóa `location` thành `location_city`.
5. Chuẩn hóa `job_industry` thành `primary_industry`.
6. Parse `salary` thành các biến có cấu trúc.
7. Tạo text length features.
8. Lưu `clean_data_train.csv` và `clean_data_test.csv`.
9. Lưu các bảng metadata cần thiết cho báo cáo/slide.

In [15]:
from pathlib import Path
import re
import unicodedata
import shutil

import numpy as np
import pandas as pd

## 1. Cấu hình đường dẫn

In [16]:
PROJECT_ROOT = Path("..").resolve()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
AUDIT_DIR = OUTPUT_DIR / "audit_stage_03"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

if AUDIT_DIR.exists():
    shutil.rmtree(AUDIT_DIR)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

raw_train_path = RAW_DIR / "raw_data_train.csv"
raw_test_path = RAW_DIR / "raw_data_test.csv"

print("Project root:", PROJECT_ROOT)
print("Raw train:", raw_train_path)
print("Raw test:", raw_test_path)

Project root: D:\DataScientFinalProject
Raw train: D:\DataScientFinalProject\data\raw\raw_data_train.csv
Raw test: D:\DataScientFinalProject\data\raw\raw_data_test.csv


## 2. Load raw train/test

In [17]:
raw_train = pd.read_csv(raw_train_path)
raw_test = pd.read_csv(raw_test_path)

print("Raw train shape:", raw_train.shape)
print("Raw test shape:", raw_test.shape)

Raw train shape: (546190, 14)
Raw test shape: (60688, 14)


## 3. Kiểm tra schema, duplicate và overlap train/test

Yêu cầu quan trọng: tập train và test không được trùng `id`.

In [18]:
same_columns = list(raw_train.columns) == list(raw_test.columns)

duplicate_summary = {
    "same_columns": same_columns,
    "raw_train_duplicate_rows": int(raw_train.duplicated().sum()),
    "raw_test_duplicate_rows": int(raw_test.duplicated().sum()),
}

if "id" in raw_train.columns and "id" in raw_test.columns:
    duplicate_summary["raw_train_duplicated_id"] = int(raw_train["id"].duplicated().sum())
    duplicate_summary["raw_test_duplicated_id"] = int(raw_test["id"].duplicated().sum())
    overlap_ids_after_split = set(raw_train["id"]).intersection(set(raw_test["id"]))
    duplicate_summary["overlap_ids_between_raw_train_test"] = len(overlap_ids_after_split)
else:
    overlap_ids_after_split = set()
    duplicate_summary["raw_train_duplicated_id"] = None
    duplicate_summary["raw_test_duplicated_id"] = None
    duplicate_summary["overlap_ids_between_raw_train_test"] = None

duplicate_summary_df = pd.DataFrame([duplicate_summary])
duplicate_summary_df.to_csv(
    TABLE_DIR / "stage_03_duplicate_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(duplicate_summary_df)

,same_columns,raw_train_duplicate_rows,raw_test_duplicate_rows,raw_train_duplicated_id,raw_test_duplicated_id,overlap_ids_between_raw_train_test
0,True,0,0,0,0,0


## 4. Làm sạch text columns

Các cột text được chuẩn hóa Unicode, chuyển về lowercase, xóa ký tự điều khiển và chuẩn hóa khoảng trắng.

In [19]:
TEXT_COLUMNS = [
    "job_title",
    "job_description",
    "requirements",
    "benefits",
]

def normalize_unicode(text):
    if pd.isna(text):
        return ""
    return unicodedata.normalize("NFC", str(text))

def clean_basic_text(text):
    text = normalize_unicode(text)
    text = text.lower()
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_text_columns(df):
    df = df.copy()
    for col in TEXT_COLUMNS:
        if col in df.columns:
            df[col] = df[col].fillna("").apply(clean_basic_text)
    return df

clean_train = clean_text_columns(raw_train)
clean_test = clean_text_columns(raw_test)

## 5. Làm sạch categorical columns

In [20]:
CATEGORICAL_COLUMNS = [
    "company_name",
    "location",
    "job_type",
    "job_industry",
    "experience_level",
    "education_level",
    "job_position",
]

def clean_categorical_value(value):
    if pd.isna(value):
        return "Unknown"

    value = unicodedata.normalize("NFC", str(value))
    value = re.sub(r"[\r\n\t]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    if value == "":
        return "Unknown"

    return value

def clean_categorical_columns(df):
    df = df.copy()
    for col in CATEGORICAL_COLUMNS:
        if col in df.columns:
            df[col] = df[col].apply(clean_categorical_value)
    return df

clean_train = clean_categorical_columns(clean_train)
clean_test = clean_categorical_columns(clean_test)

## 6. Chuẩn hóa `location` thành `location_city`

Cột `location` gốc là địa chỉ tự do, có thể chứa số nhà, đường, phường/xã, quận/huyện và tỉnh/thành. Vì vậy, notebook tạo thêm `location_city` để gom về cấp tỉnh/thành phục vụ EDA và feature engineering.

Các địa chỉ không nhận diện được sẽ được gán `Khác/Không rõ`, không xóa dữ liệu.

In [21]:
def remove_vietnamese_accents(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = unicodedata.normalize("NFD", text)
    text = "".join(
        ch for ch in text
        if unicodedata.category(ch) != "Mn"
    )
    text = text.replace("Đ", "D").replace("đ", "d")
    return text

def normalize_location_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()
    text = remove_vietnamese_accents(text).lower()
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

CITY_PATTERNS = {
    "TP Hồ Chí Minh": [
        "tp ho chi minh", "thanh pho ho chi minh", "ho chi minh",
        "tphcm", "tp hcm", "hcm", "sai gon", "saigon",

        # Quận cũ và ký hiệu viết tắt
        "quan 1", "quan 3", "quan 4", "quan 5", "quan 6", "quan 7",
        "quan 8", "quan 10", "quan 11", "quan 12",
        "q1", "q 1", "q3", "q 3", "q4", "q 4", "q5", "q 5",
        "q6", "q 6", "q7", "q 7", "q8", "q 8",
        "q10", "q 10", "q11", "q 11", "q12", "q 12",

        # Quận/huyện phổ biến
        "tan binh", "tan phu", "binh tan", "binh thanh",
        "phu nhuan", "go vap", "thu duc", "tp thu duc",
        "nha be", "binh chanh", "hoc mon", "cu chi", "can gio",

        # Một số phường/khu vực thường gặp trong tin tuyển dụng
        "ben nghe", "ben thanh", "thao dien", "an phu",
        "tan phong", "tan thuan", "thanh my loi", "thu thiem",
        "pham phu thu", "hoang van thu", "cong hoa", "truong chinh",
        "nguyen van troi", "dien bien phu", "le van sy"
    ],
    "Hà Nội": [
        "ha noi", "thanh pho ha noi", "hn",

        # Quận/huyện phổ biến
        "dong da", "ba dinh", "hoan kiem", "hai ba trung",
        "cau giay", "thanh xuan", "hoang mai", "long bien",
        "tay ho", "nam tu liem", "bac tu liem", "ha dong",
        "gia lam", "dong anh", "thanh tri", "hoai duc",
        "dan phuong", "thach that", "quoc oai", "me linh",
        "soc son", "chuong my", "thuong tin", "phuc tho",
        "son tay",

        # Một số khu vực/đường thường gặp
        "to huu", "hoang cau", "trung van", "my dinh",
        "cau dien", "nguyen trai", "lang ha", "kim ma",
        "tran duy hung", "duy tan", "pham hung", "linh dam"
    ],
    "Đà Nẵng": [
        "da nang", "thanh pho da nang"
    ],
    "Hải Phòng": [
        "hai phong", "thanh pho hai phong"
    ],
    "Cần Thơ": [
        "can tho", "thanh pho can tho"
    ],
    "Huế": [
        "hue", "thua thien hue"
    ],
    "Bình Dương": ["binh duong"],
    "Đồng Nai": ["dong nai"],
    "Bắc Ninh": ["bac ninh"],
    "Long An": ["long an"],
    "Hưng Yên": ["hung yen"],
    "Thanh Hóa": ["thanh hoa"],
    "Nghệ An": ["nghe an"],
    "Khánh Hòa": ["khanh hoa", "nha trang"],
    "Bà Rịa - Vũng Tàu": ["ba ria", "vung tau", "ba ria vung tau"],
    "Lâm Đồng": ["lam dong", "da lat"],
    "Quảng Ninh": ["quang ninh", "ha long", "cam pha"],
    "Tiền Giang": ["tien giang"],
    "An Giang": ["an giang"],
    "Kiên Giang": ["kien giang", "phu quoc", "rach gia"],
    "Quảng Nam": ["quang nam", "hoi an", "tam ky"],
    "Quảng Ngãi": ["quang ngai"],
    "Bình Định": ["binh dinh", "quy nhon"],
    "Phú Thọ": ["phu tho", "viet tri"],
    "Vĩnh Phúc": ["vinh phuc", "vinh yen"],
    "Thái Nguyên": ["thai nguyen"],
    "Nam Định": ["nam dinh"],
    "Ninh Bình": ["ninh binh"],
    "Hải Dương": ["hai duong"],
    "Hà Giang": ["ha giang"],
    "Cao Bằng": ["cao bang"],
    "Bắc Kạn": ["bac kan"],
    "Tuyên Quang": ["tuyen quang"],
    "Lào Cai": ["lao cai", "sapa", "sa pa"],
    "Điện Biên": ["dien bien"],
    "Lai Châu": ["lai chau"],
    "Sơn La": ["son la"],
    "Yên Bái": ["yen bai"],
    "Hòa Bình": ["hoa binh"],
    "Lạng Sơn": ["lang son"],
    "Bắc Giang": ["bac giang"],
    "Thái Bình": ["thai binh"],
    "Hà Nam": ["ha nam"],
    "Hà Tĩnh": ["ha tinh"],
    "Quảng Bình": ["quang binh"],
    "Quảng Trị": ["quang tri"],
    "Bình Thuận": ["binh thuan", "phan thiet"],
    "Ninh Thuận": ["ninh thuan", "phan rang"],
    "Gia Lai": ["gia lai", "pleiku", "plei ku"],
    "Kon Tum": ["kon tum"],
    "Đắk Lắk": ["dak lak", "dac lac", "buon ma thuot", "buon me thuot"],
    "Đắk Nông": ["dak nong", "dac nong"],
    "Bình Phước": ["binh phuoc"],
    "Tây Ninh": ["tay ninh"],
    "Bến Tre": ["ben tre"],
    "Trà Vinh": ["tra vinh"],
    "Vĩnh Long": ["vinh long"],
    "Đồng Tháp": ["dong thap", "sa dec", "cao lanh"],
    "Hậu Giang": ["hau giang"],
    "Sóc Trăng": ["soc trang"],
    "Bạc Liêu": ["bac lieu"],
    "Cà Mau": ["ca mau"],
    "Phú Yên": ["phu yen", "tuy hoa"],
}

CITY_PATTERN_ITEMS = []
for city, patterns in CITY_PATTERNS.items():
    for pattern in patterns:
        CITY_PATTERN_ITEMS.append((city, normalize_location_text(pattern)))

CITY_PATTERN_ITEMS = sorted(
    CITY_PATTERN_ITEMS,
    key=lambda x: len(x[1]),
    reverse=True
)

def extract_city_from_location(location):
    text = normalize_location_text(location)

    if text == "" or text == "unknown":
        return "Khác/Không rõ"

    for city, pattern in CITY_PATTERN_ITEMS:
        if re.search(rf"(^|[^a-z0-9]){re.escape(pattern)}([^a-z0-9]|$)", text):
            return city

    return "Khác/Không rõ"

clean_train["location_city"] = clean_train["location"].apply(extract_city_from_location)
clean_test["location_city"] = clean_test["location"].apply(extract_city_from_location)

clean_train["location_city_is_unknown"] = (
    clean_train["location_city"] == "Khác/Không rõ"
).astype(int)

clean_test["location_city_is_unknown"] = (
    clean_test["location_city"] == "Khác/Không rõ"
).astype(int)

location_city_coverage = (
    clean_train["location_city"]
    .value_counts(dropna=False)
    .reset_index()
)
location_city_coverage.columns = ["location_city", "count"]
location_city_coverage["ratio"] = location_city_coverage["count"] / len(clean_train)

location_city_coverage.to_csv(
    TABLE_DIR / "stage_03_location_city_coverage.csv",
    index=False,
    encoding="utf-8-sig"
)

unknown_location_count = int((clean_train["location_city"] == "Khác/Không rõ").sum())
unknown_location_ratio = clean_train["location_city"].eq("Khác/Không rõ").mean()

unknown_location_sample = clean_train[
    clean_train["location_city"] == "Khác/Không rõ"
][["location", "location_city"]].sample(
    n=min(100, unknown_location_count),
    random_state=42
)

unknown_location_sample.to_csv(
    TABLE_DIR / "stage_03_unknown_location_sample_100.csv",
    index=False,
    encoding="utf-8-sig"
)

display(location_city_coverage.head(20))
print("Unknown location_city count:", unknown_location_count)
print("Unknown location_city ratio:", unknown_location_ratio)

,location_city,count,ratio
0,TP Hồ Chí Minh,285382,0.522496
1,Hà Nội,141002,0.258156
2,Bình Dương,25368,0.046445
3,Khác/Không rõ,13455,0.024634
4,Đồng Nai,10150,0.018583
5,Đà Nẵng,8277,0.015154
6,Long An,7148,0.013087
7,Bắc Ninh,4823,0.008830
8,Hải Phòng,4542,0.008316
9,Hưng Yên,4451,0.008149


Unknown location_city count: 13455
Unknown location_city ratio: 0.024634284772698144


## 7. Chuẩn hóa `job_industry` thành `primary_industry`

Cột `job_industry` có thể chứa nhiều ngành trong một chuỗi. Notebook tạo thêm `primary_industry` bằng cách lấy ngành đầu tiên để phục vụ EDA và feature engineering.

In [22]:
def extract_primary_industry(industry):
    if pd.isna(industry):
        return "Unknown"

    text = str(industry).strip()

    if text == "" or text.lower() == "unknown":
        return "Unknown"

    primary = text.split(",")[0].strip()

    if primary == "":
        return "Unknown"

    return primary

clean_train["primary_industry"] = clean_train["job_industry"].apply(extract_primary_industry)
clean_test["primary_industry"] = clean_test["job_industry"].apply(extract_primary_industry)

primary_industry_coverage = (
    clean_train["primary_industry"]
    .value_counts(dropna=False)
    .reset_index()
)
primary_industry_coverage.columns = ["primary_industry", "count"]
primary_industry_coverage["ratio"] = primary_industry_coverage["count"] / len(clean_train)

primary_industry_coverage.to_csv(
    TABLE_DIR / "stage_03_primary_industry_coverage.csv",
    index=False,
    encoding="utf-8-sig"
)

display(primary_industry_coverage.head(20))

,primary_industry,count,ratio
0,Bán hàng - Kinh doanh,52977,0.096994
1,Chăm sóc khách hàng,50531,0.092515
2,Kế toán / Kiểm toán,45469,0.083248
3,Xây dựng,27020,0.049470
4,Chưa xác định,23057,0.042214
5,Cơ khí - Ô tô - Tự động hóa / Sản xuất - Lắp r...,22371,0.040958
6,Khoa học - Kỹ thuật,21595,0.039538
7,Lao động phổ thông,21068,0.038573
8,Nghề nghiệp khác,20985,0.038421
9,Giáo dục - Đào tạo / Biên phiên dịch,16638,0.030462


## 8. Parse salary

Cột `salary` được chuyển thành các biến có cấu trúc. Các salary bất thường được giữ lại nhưng đánh dấu bằng `salary_parse_issue`.

In [23]:
def parse_number_token(token):
    token = str(token).strip()

    if "." in token:
        parts = token.split(".")
        if len(parts) > 1 and all(len(part) == 3 for part in parts[1:]):
            return float(token.replace(".", ""))

    if "," in token:
        parts = token.split(",")
        if len(parts) > 1 and all(len(part) == 3 for part in parts[1:]):
            return float(token.replace(",", ""))

    token = token.replace(",", ".")
    return float(token)

def parse_salary_to_million_vnd(salary):
    if pd.isna(salary):
        return pd.Series({
            "salary_available": 0,
            "salary_min_million_vnd": np.nan,
            "salary_max_million_vnd": np.nan,
            "salary_avg_million_vnd": np.nan,
            "salary_currency": "Unknown"
        })

    text = str(salary).strip().lower()

    unavailable_keywords = [
        "thỏa thuận",
        "thoả thuận",
        "thương lượng",
        "tuỳ năng lực",
        "tùy năng lực",
        "cạnh tranh",
        "đang cập nhật",
        "không công khai",
        "vnd month",
        "you'll love it",
        "unknown"
    ]

    if text == "" or any(keyword in text for keyword in unavailable_keywords):
        return pd.Series({
            "salary_available": 0,
            "salary_min_million_vnd": np.nan,
            "salary_max_million_vnd": np.nan,
            "salary_avg_million_vnd": np.nan,
            "salary_currency": "Unknown"
        })

    currency = "VND"
    if "usd" in text or "$" in text:
        currency = "USD"

    numbers = re.findall(r"\d+(?:[.,]\d+)*", text)

    if len(numbers) == 0:
        return pd.Series({
            "salary_available": 0,
            "salary_min_million_vnd": np.nan,
            "salary_max_million_vnd": np.nan,
            "salary_avg_million_vnd": np.nan,
            "salary_currency": "Unknown"
        })

    parsed_numbers = []
    for token in numbers:
        try:
            parsed_numbers.append(parse_number_token(token))
        except Exception:
            pass

    if len(parsed_numbers) == 0:
        return pd.Series({
            "salary_available": 0,
            "salary_min_million_vnd": np.nan,
            "salary_max_million_vnd": np.nan,
            "salary_avg_million_vnd": np.nan,
            "salary_currency": "Unknown"
        })

    if currency == "USD":
        return pd.Series({
            "salary_available": 1,
            "salary_min_million_vnd": np.nan,
            "salary_max_million_vnd": np.nan,
            "salary_avg_million_vnd": np.nan,
            "salary_currency": "USD"
        })

    converted = []

    for value in parsed_numbers:
        if value >= 1_000_000:
            converted.append(value / 1_000_000)
        elif any(unit in text for unit in ["triệu", "trieu", " tr", "tr "]):
            converted.append(value)
        elif any(unit in text for unit in ["vnd", "vnđ", "đ"]):
            converted.append(value / 1_000_000)
        else:
            converted.append(value)

    if len(converted) == 0:
        return pd.Series({
            "salary_available": 0,
            "salary_min_million_vnd": np.nan,
            "salary_max_million_vnd": np.nan,
            "salary_avg_million_vnd": np.nan,
            "salary_currency": "Unknown"
        })

    salary_min = min(converted)
    salary_max = max(converted)
    salary_avg = (salary_min + salary_max) / 2

    return pd.Series({
        "salary_available": 1,
        "salary_min_million_vnd": salary_min,
        "salary_max_million_vnd": salary_max,
        "salary_avg_million_vnd": salary_avg,
        "salary_currency": "VND"
    })

def add_salary_features(df):
    df = df.copy()

    if "salary" in df.columns:
        df["salary"] = df["salary"].fillna("Unknown")
        salary_features = df["salary"].apply(parse_salary_to_million_vnd)
        df = pd.concat([df, salary_features], axis=1)

        salary_text = df["salary"].astype(str).str.strip().str.lower()

        unavailable_pattern = (
            r"thỏa thuận|thoả thuận|thương lượng|tuỳ năng lực|tùy năng lực|"
            r"cạnh tranh|đang cập nhật|không công khai|vnd month|you'll love it|unknown"
        )

        is_unavailable = (
            (salary_text == "")
            | salary_text.str.contains(unavailable_pattern, regex=True, na=False)
        )

        has_number = salary_text.str.contains(r"\d", regex=True, na=False)

        df["salary_parse_issue"] = (
            (~is_unavailable)
            & has_number
            & (
                (df["salary_currency"] == "VND")
                & (
                    df["salary_avg_million_vnd"].isna()
                    | (df["salary_min_million_vnd"] <= 0)
                    | (df["salary_max_million_vnd"] <= 0)
                    | (df["salary_min_million_vnd"] > df["salary_max_million_vnd"])
                    | (df["salary_avg_million_vnd"] > 200)
                )
            )
        ).astype(int)

    return df

clean_train = add_salary_features(clean_train)
clean_test = add_salary_features(clean_test)

## 9. Làm sạch year và tạo text length features

In [24]:
def clean_year_column(df):
    df = df.copy()
    if "year" in df.columns:
        df["year"] = pd.to_numeric(df["year"], errors="coerce")
    return df

def add_text_length_features(df):
    df = df.copy()
    for col in TEXT_COLUMNS:
        if col in df.columns:
            df[f"{col}_char_len"] = df[col].str.len()
            df[f"{col}_word_count"] = df[col].str.split().apply(len)
    return df

clean_train = clean_year_column(clean_train)
clean_test = clean_year_column(clean_test)

clean_train = add_text_length_features(clean_train)
clean_test = add_text_length_features(clean_test)

## 10. Kiểm tra sau cleaning và lưu bảng thống kê

In [25]:
if "id" in clean_train.columns and "id" in clean_test.columns:
    overlap_ids_after_clean = set(clean_train["id"]).intersection(set(clean_test["id"]))
else:
    overlap_ids_after_clean = set()

clean_missing_summary = pd.DataFrame({
    "column": clean_train.columns,
    "missing_count": clean_train.isna().sum().values,
    "missing_ratio": clean_train.isna().mean().values,
}).sort_values("missing_ratio", ascending=False)

clean_missing_summary.to_csv(
    TABLE_DIR / "stage_03_missing_summary_clean_train.csv",
    index=False,
    encoding="utf-8-sig"
)

salary_anomalies = clean_train[
    clean_train["salary_parse_issue"] == 1
][
    [
        "salary",
        "salary_available",
        "salary_min_million_vnd",
        "salary_max_million_vnd",
        "salary_avg_million_vnd",
        "salary_currency",
        "salary_parse_issue",
    ]
]

salary_anomalies.to_csv(
    TABLE_DIR / "stage_03_salary_anomalies.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Clean train shape:", clean_train.shape)
print("Clean test shape:", clean_test.shape)
print("Overlap IDs after clean:", len(overlap_ids_after_clean))
print("Salary parse issue train ratio:", clean_train["salary_parse_issue"].mean())
print("Unknown location_city ratio:", clean_train["location_city"].eq("Khác/Không rõ").mean())

Clean train shape: (546190, 31)
Clean test shape: (60688, 31)
Overlap IDs after clean: 0
Salary parse issue train ratio: 0.014075687947417564
Unknown location_city ratio: 0.024634284772698144


## 11. Lưu clean data

In [26]:
clean_train_path = CLEAN_DIR / "clean_data_train.csv"
clean_test_path = CLEAN_DIR / "clean_data_test.csv"

clean_train.to_csv(clean_train_path, index=False, encoding="utf-8-sig")
clean_test.to_csv(clean_test_path, index=False, encoding="utf-8-sig")

clean_train.to_csv(PROJECT_ROOT / "clean_data_train.csv", index=False, encoding="utf-8-sig")
clean_test.to_csv(PROJECT_ROOT / "clean_data_test.csv", index=False, encoding="utf-8-sig")

print("Saved clean train:", clean_train_path)
print("Saved clean test:", clean_test_path)
print("Also copied clean CSV files to project root.")

Saved clean train: D:\DataScientFinalProject\data\clean\clean_data_train.csv
Saved clean test: D:\DataScientFinalProject\data\clean\clean_data_test.csv
Also copied clean CSV files to project root.


## 12. Lưu metadata và summary tables

In [27]:
stage_03_metadata = {
    "n_raw_train": len(raw_train),
    "n_clean_train": len(clean_train),
    "n_raw_test": len(raw_test),
    "n_clean_test": len(clean_test),

    "n_columns_raw_train": raw_train.shape[1],
    "n_columns_clean_train": clean_train.shape[1],

    "overlap_ids_after_clean": len(overlap_ids_after_clean),

    "salary_available_train_ratio": clean_train["salary_available"].mean(),
    "salary_available_test_ratio": clean_test["salary_available"].mean(),

    "salary_parse_issue_train_count": int(clean_train["salary_parse_issue"].sum()),
    "salary_parse_issue_test_count": int(clean_test["salary_parse_issue"].sum()),

    "salary_parse_issue_train_ratio": clean_train["salary_parse_issue"].mean(),
    "salary_parse_issue_test_ratio": clean_test["salary_parse_issue"].mean(),

    "n_unique_location_city_train": clean_train["location_city"].nunique(),
    "n_unique_primary_industry_train": clean_train["primary_industry"].nunique(),

    "top_location_city_train": clean_train["location_city"].value_counts().index[0],
    "top_primary_industry_train": clean_train["primary_industry"].value_counts().index[0],

    "unknown_location_city_count": int((clean_train["location_city"] == "Khác/Không rõ").sum()),
    "unknown_location_city_ratio": clean_train["location_city"].eq("Khác/Không rõ").mean(),
    
    "location_city_known_count": int((clean_train["location_city"] != "Khác/Không rõ").sum()) if "location_city" in clean_train.columns else None,
    "location_city_known_ratio": clean_train["location_city"].ne("Khác/Không rõ").mean() if "location_city" in clean_train.columns else None,
}

stage_03_metadata_df = pd.DataFrame([stage_03_metadata])
stage_03_metadata_df.to_csv(
    TABLE_DIR / "stage_03_metadata.csv",
    index=False,
    encoding="utf-8-sig"
)

salary_available_counts = (
    clean_train["salary_available"]
    .value_counts(dropna=False)
    .rename_axis("salary_available")
    .reset_index(name="count")
)
salary_available_counts["ratio"] = salary_available_counts["count"] / len(clean_train)

salary_currency_counts = (
    clean_train["salary_currency"]
    .value_counts(dropna=False)
    .rename_axis("salary_currency")
    .reset_index(name="count")
)
salary_currency_counts["ratio"] = salary_currency_counts["count"] / len(clean_train)

salary_numeric_cols = [
    "salary_min_million_vnd",
    "salary_max_million_vnd",
    "salary_avg_million_vnd",
]

salary_numeric_describe_all = (
    clean_train[salary_numeric_cols]
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "column"})
)

salary_numeric_describe_no_issue = (
    clean_train[clean_train["salary_parse_issue"] == 0][salary_numeric_cols]
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "column"})
)

salary_available_counts.to_csv(
    TABLE_DIR / "stage_03_salary_available_counts.csv",
    index=False,
    encoding="utf-8-sig"
)

salary_currency_counts.to_csv(
    TABLE_DIR / "stage_03_salary_currency_counts.csv",
    index=False,
    encoding="utf-8-sig"
)

salary_numeric_describe_all.to_csv(
    TABLE_DIR / "stage_03_salary_numeric_describe_all.csv",
    index=False,
    encoding="utf-8-sig"
)

salary_numeric_describe_no_issue.to_csv(
    TABLE_DIR / "stage_03_salary_numeric_describe_no_issue.csv",
    index=False,
    encoding="utf-8-sig"
)

display(stage_03_metadata_df)

,n_raw_train,n_clean_train,n_raw_test,n_clean_test,n_columns_raw_train,n_columns_clean_train,overlap_ids_after_clean,salary_available_train_ratio,salary_available_test_ratio,salary_parse_issue_train_count,...,salary_parse_issue_train_ratio,salary_parse_issue_test_ratio,n_unique_location_city_train,n_unique_primary_industry_train,top_location_city_train,top_primary_industry_train,unknown_location_city_count,unknown_location_city_ratio,location_city_known_count,location_city_known_ratio
0,546190,546190,60688,60688,14,31,0,0.91697,0.916062,7688,...,0.014076,0.013825,64,2783,TP Hồ Chí Minh,Bán hàng - Kinh doanh,13455,0.024634,532735,0.975366


## 13. Lưu audit files cần thiết

Audit files nhỏ giúp kiểm tra dữ liệu clean mà không cần gửi toàn bộ CSV.

In [28]:
clean_sample_1000 = clean_train.sample(
    n=min(1000, len(clean_train)),
    random_state=42
)

clean_sample_1000.to_csv(
    AUDIT_DIR / "stage_03_clean_data_sample_1000.csv",
    index=False,
    encoding="utf-8-sig"
)

schema_df = pd.DataFrame({
    "column": clean_train.columns,
    "dtype": [str(dtype) for dtype in clean_train.dtypes],
    "missing_count": clean_train.isna().sum().values,
    "missing_ratio": clean_train.isna().mean().values,
})

schema_df.to_csv(
    AUDIT_DIR / "stage_03_clean_schema.csv",
    index=False,
    encoding="utf-8-sig"
)

salary_parse_sample = clean_train[
    [
        "salary",
        "salary_available",
        "salary_min_million_vnd",
        "salary_max_million_vnd",
        "salary_avg_million_vnd",
        "salary_currency",
        "salary_parse_issue",
    ]
].sample(
    n=min(200, len(clean_train)),
    random_state=42
)

salary_parse_sample.to_csv(
    AUDIT_DIR / "stage_03_salary_parse_sample_200.csv",
    index=False,
    encoding="utf-8-sig"
)

text_clean_sample = clean_train[
    [
        "job_title",
        "job_description",
        "requirements",
        "benefits",
    ]
].sample(
    n=min(200, len(clean_train)),
    random_state=42
)

text_clean_sample.to_csv(
    AUDIT_DIR / "stage_03_text_clean_sample_200.csv",
    index=False,
    encoding="utf-8-sig"
)

numeric_cols = [
    "salary_min_million_vnd",
    "salary_max_million_vnd",
    "salary_avg_million_vnd",
    "salary_parse_issue",
    "location_city_is_unknown",
    "job_title_char_len",
    "job_title_word_count",
    "job_description_char_len",
    "job_description_word_count",
    "requirements_char_len",
    "requirements_word_count",
    "benefits_char_len",
    "benefits_word_count",
    "year",
]

existing_numeric_cols = [col for col in numeric_cols if col in clean_train.columns]

numeric_describe = (
    clean_train[existing_numeric_cols]
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "column"})
)

numeric_describe.to_csv(
    AUDIT_DIR / "stage_03_numeric_describe.csv",
    index=False,
    encoding="utf-8-sig"
)

categorical_cols = [
    "company_name",
    "location",
    "location_city",
    "job_type",
    "job_industry",
    "primary_industry",
    "experience_level",
    "education_level",
    "job_position",
    "salary_currency",
]

for col in categorical_cols:
    if col in clean_train.columns:
        vc = (
            clean_train[col]
            .value_counts(dropna=False)
            .head(100)
            .rename_axis(col)
            .reset_index(name="count")
        )

        vc["ratio"] = vc["count"] / len(clean_train)

        vc.to_csv(
            AUDIT_DIR / f"stage_03_value_counts_{col}.csv",
            index=False,
            encoding="utf-8-sig"
        )

salary_anomalies.to_csv(
    AUDIT_DIR / "stage_03_salary_anomalies.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved audit files to:", AUDIT_DIR)

Saved audit files to: D:\DataScientFinalProject\outputs\audit_stage_03


## 14. Kết luận

Sau khi chạy xong notebook, cần kiểm tra nhanh:

```text
clean_train shape = (546190, 30)
clean_test shape  = (60688, 30)
overlap_ids_after_clean = 0
```

Các file chính cần có:

```text
data/clean/clean_data_train.csv
data/clean/clean_data_test.csv
outputs/tables/stage_03_metadata.csv
outputs/tables/stage_03_location_city_coverage.csv
outputs/tables/stage_03_primary_industry_coverage.csv
outputs/tables/stage_03_missing_summary_clean_train.csv
outputs/audit_stage_03/
```